# E-Commerce Customer Intelligence Analysis

**Objective:** Identify revenue concentration, customer value tiers, repeat-purchase behavior, promotion patterns, and product/category opportunities from customer shopping data.

**Workflow:** CSV → Data Quality → Cleaning → Feature Engineering → EDA → SQL Business Analysis → Power BI


In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_shopping_behavior.csv")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")


Rows: 3,900 | Columns: 18


In [21]:
# Data-quality review
print(df.head())
print("\nShape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])


   Customer ID  Age Gender Item Purchased  Category  Purchase Amount (USD)  \
0            1   55   Male         Blouse  Clothing                     53   
1            2   19   Male        Sweater  Clothing                     64   
2            3   50   Male          Jeans  Clothing                     73   
3            4   21   Male        Sandals  Footwear                     90   
4            5   45   Male         Blouse  Clothing                     49   

        Location Size      Color  Season  Review Rating Subscription Status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   Shipping Type Discount Applied Promo Code Used  Previ

In [22]:
# Cleaning
# Review ratings are the only missing analytical field in this dataset.
# Use category-level medians so the imputation respects category-specific rating patterns.
df["Review Rating"] = (
    df.groupby("Category")["Review Rating"]
      .transform(lambda s: s.fillna(s.median()))
)

# Standardize names for consistent Python and SQL usage.
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_")
)
df = df.rename(columns={"purchase_amount_usd": "purchase_amount"})

print("Remaining missing values:", int(df.isna().sum().sum()))


Remaining missing values: 0


## Feature Engineering

In [23]:
# 1) Business-friendly age bands
age_bins = [-1, 24, 34, 44, np.inf]
age_labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)

# 2) Convert purchase cadence into an approximate number of days.
frequency_days = {
    "Weekly": 7, "Fortnightly": 14, "Bi-Weekly": 14,
    "Monthly": 30, "Every 3 Months": 90, "Quarterly": 90, "Annually": 365
}
df["purchase_frequency_days"] = df["frequency_of_purchases"].map(frequency_days)

# 3) Transparent behavioral segmentation using purchase history.
df["customer_segment"] = pd.cut(
    df["previous_purchases"],
    bins=[-np.inf, 5, 20, np.inf],
    labels=["New / Low-Repeat", "Returning", "Loyal"]
)

# 4) A separate value tier based on the current transaction amount.
# qcut creates four equally populated groups, avoiding arbitrary dollar cutoffs.
df["value_tier"] = pd.qcut(
    df["purchase_amount"],
    q=4,
    labels=["Entry", "Core", "Premium", "High Value"]
)

# 5) Combine promotion and subscription status into a business audience flag.
df["promotion_audience"] = np.select(
    [
        (df["discount_applied"] == "Yes") & (df["subscription_status"] == "Yes"),
        (df["discount_applied"] == "Yes") & (df["subscription_status"] == "No")
    ],
    ["Discounted Subscriber", "Discounted Non-Subscriber"],
    default="No Discount"
)

# Promo-code usage is highly redundant with discount status for this dataset.
if "promo_code_used" in df.columns:
    df = df.drop(columns=["promo_code_used"])

df[["customer_segment", "value_tier", "promotion_audience"]].head()


,customer_segment,value_tier,promotion_audience
0,Returning,Core,Discounted Subscriber
1,New / Low-Repeat,Premium,Discounted Subscriber
2,Loyal,Premium,Discounted Subscriber
3,Loyal,High Value,Discounted Subscriber
4,Loyal,Core,Discounted Subscriber


## Executive KPIs

In [24]:
kpis = {
    "Total Revenue": df["purchase_amount"].sum(),
    "Transactions": len(df),
    "Average Order Value": df["purchase_amount"].mean(),
    "Unique Customers": df["customer_id"].nunique(),
    "Subscribed Customers": (df["subscription_status"] == "Yes").sum(),
    "Discounted Orders": (df["discount_applied"] == "Yes").sum()
}
pd.Series(kpis).round(2)


Total Revenue           233081.00
Transactions              3900.00
Average Order Value         59.76
Unique Customers          3900.00
Subscribed Customers      1053.00
Discounted Orders         1677.00
dtype: float64

## Revenue Concentration

In [25]:
category_summary = (
    df.groupby("category")
      .agg(revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           transactions=("customer_id", "count"))
      .sort_values("revenue", ascending=False)
)
category_summary["revenue_share_pct"] = category_summary["revenue"] / df["purchase_amount"].sum() * 100
category_summary.round(2)


,revenue,avg_order_value,transactions,revenue_share_pct
category,,,,
Clothing,104264,60.03,1737,44.73
Accessories,74200,59.84,1240,31.83
Footwear,36093,60.26,599,15.49
Outerwear,18524,57.17,324,7.95


In [26]:
location_summary = (
    df.groupby("location")
      .agg(revenue=("purchase_amount", "sum"),
           transactions=("customer_id", "count"),
           avg_order_value=("purchase_amount", "mean"))
      .sort_values("revenue", ascending=False)
)
location_summary.head(10).round(2)


,revenue,transactions,avg_order_value
location,,,
Montana,5784,96,60.25
Illinois,5617,92,61.05
California,5605,95,59.00
Idaho,5587,93,60.08
Nevada,5514,87,63.38
Alabama,5261,89,59.11
New York,5257,87,60.43
North Dakota,5220,83,62.89
West Virginia,5174,81,63.88


## Product & Rating Analysis

In [27]:
item_summary = (
    df.groupby("item_purchased")
      .agg(revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           transactions=("customer_id", "count"),
           avg_rating=("review_rating", "mean"))
      .sort_values("revenue", ascending=False)
)
item_summary["revenue_share_pct"] = item_summary["revenue"] / df["purchase_amount"].sum() * 100
item_summary.head(10).round(2)


,revenue,avg_order_value,transactions,avg_rating,revenue_share_pct
item_purchased,,,,,
Blouse,10410,60.88,171,3.68,4.47
Shirt,10332,61.14,169,3.62,4.43
Dress,10320,62.17,166,3.75,4.43
Pants,10090,59.01,171,3.72,4.33
Jewelry,10010,58.54,171,3.76,4.29
Sunglasses,9649,59.93,161,3.75,4.14
Belt,9635,59.84,161,3.76,4.13
Scarf,9561,60.90,157,3.71,4.10
Sweater,9462,57.70,164,3.76,4.06


In [28]:
# Identify products that combine commercial performance with strong customer ratings.
product_quality = item_summary.query("transactions >= 50").copy()
product_quality["rating_revenue_score"] = product_quality["avg_rating"] * product_quality["revenue_share_pct"]
product_quality.sort_values("rating_revenue_score", ascending=False).head(10).round(2)


,revenue,avg_order_value,transactions,avg_rating,revenue_share_pct,rating_revenue_score
item_purchased,,,,,,
Dress,10320,62.17,166,3.75,4.43,16.60
Blouse,10410,60.88,171,3.68,4.47,16.44
Jewelry,10010,58.54,171,3.76,4.29,16.14
Pants,10090,59.01,171,3.72,4.33,16.11
Shirt,10332,61.14,169,3.62,4.43,16.06
Belt,9635,59.84,161,3.76,4.13,15.56
Sunglasses,9649,59.93,161,3.75,4.14,15.53
Hat,9375,60.88,154,3.80,4.02,15.29
Skirt,9402,59.51,158,3.78,4.03,15.27


## Customer Intelligence

In [29]:
segment_summary = (
    df.groupby("customer_segment", observed=False)
      .agg(customers=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           avg_previous_purchases=("previous_purchases", "mean"))
)
segment_summary["revenue_share_pct"] = segment_summary["revenue"] / df["purchase_amount"].sum() * 100
segment_summary.round(2)


,customers,revenue,avg_order_value,avg_previous_purchases,revenue_share_pct
customer_segment,,,,,
New / Low-Repeat,424,25692,60.59,3.06,11.02
Returning,1137,67756,59.59,13.11,29.07
Loyal,2339,139633,59.70,35.34,59.91


In [30]:
value_tier_summary = (
    df.groupby("value_tier", observed=False)
      .agg(customers=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_previous_purchases=("previous_purchases", "mean"))
)
value_tier_summary["revenue_share_pct"] = value_tier_summary["revenue"] / df["purchase_amount"].sum() * 100
value_tier_summary.round(2)


,customers,revenue,avg_previous_purchases,revenue_share_pct
value_tier,,,,
Entry,1014,29932,25.19,12.84
Core,972,48602,25.15,20.85
Premium,988,70197,25.58,30.12
High Value,926,84350,25.49,36.19


In [31]:
subscription_summary = (
    df.groupby("subscription_status")
      .agg(customers=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"),
           avg_previous_purchases=("previous_purchases", "mean"))
)
subscription_summary["revenue_share_pct"] = subscription_summary["revenue"] / df["purchase_amount"].sum() * 100
subscription_summary.round(2)


,customers,revenue,avg_order_value,avg_previous_purchases,revenue_share_pct
subscription_status,,,,,
No,2847,170436,59.87,25.08,73.12
Yes,1053,62645,59.49,26.08,26.88


## Promotion & Basket Behavior

In [32]:
promotion_summary = (
    df.groupby("promotion_audience")
      .agg(orders=("customer_id", "count"),
           revenue=("purchase_amount", "sum"),
           avg_order_value=("purchase_amount", "mean"))
      .sort_values("revenue", ascending=False)
)
promotion_summary.round(2)


,orders,revenue,avg_order_value
promotion_audience,,,
No Discount,2223,133670,60.13
Discounted Subscriber,1053,62645,59.49
Discounted Non-Subscriber,624,36766,58.92


In [33]:
discount_by_item = (
    df.groupby("item_purchased")
      .agg(orders=("customer_id", "count"),
           discounted_orders=("discount_applied", lambda s: (s == "Yes").sum()))
)
discount_by_item["discount_rate_pct"] = discount_by_item["discounted_orders"] / discount_by_item["orders"] * 100
discount_by_item.sort_values("discount_rate_pct", ascending=False).head(10).round(2)


,orders,discounted_orders,discount_rate_pct
item_purchased,,,
Hat,154,77,50.00
Sneakers,145,72,49.66
Coat,161,79,49.07
Sweater,164,79,48.17
Pants,171,81,47.37
Boots,144,67,46.53
Jeans,124,57,45.97
Dress,166,75,45.18
Hoodie,151,68,45.03


## Season, Shipping, Payment & Age

In [34]:
season_summary = (
    df.groupby("season")
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
      .sort_values("revenue", ascending=False)
)
shipping_summary = (
    df.groupby("shipping_type")
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
      .sort_values("avg_order_value", ascending=False)
)
payment_summary = (
    df.groupby("payment_method")
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
      .sort_values("revenue", ascending=False)
)
age_summary = (
    df.groupby("age_group", observed=False)
      .agg(revenue=("purchase_amount", "sum"), avg_order_value=("purchase_amount", "mean"), transactions=("customer_id", "count"))
)
print("Season\n", season_summary.round(2))
print("\nShipping\n", shipping_summary.round(2))
print("\nPayment\n", payment_summary.round(2))
print("\nAge\n", age_summary.round(2))


Season
         revenue  avg_order_value  transactions
season                                        
Fall      60018            61.56           975
Spring    58679            58.74           999
Winter    58607            60.36           971
Summer    55777            58.41           955

Shipping
                 revenue  avg_order_value  transactions
shipping_type                                         
2-Day Shipping    38080            60.73           627
Express           39067            60.48           646
Free Shipping     40777            60.41           675
Store Pickup      38931            59.89           650
Next Day Air      37993            58.63           648
Standard          38233            58.46           654

Payment
                 revenue  avg_order_value  transactions
payment_method                                        
Credit Card       40310            60.07           671
PayPal            40109            59.25           677
Cash              40002      

## Analyst Takeaways

Focus the final dashboard on four questions: **Where is revenue concentrated? Which customer/value tiers matter most? Where are promotions being used? Which products/categories combine scale with customer satisfaction?** Avoid claiming causal effects or true retention/churn because the dataset is cross-sectional and does not contain a full transaction history over time.
